In [ ]:
# Already run once; the outputs are in Drive. Re-running rebuilds the engine (about 5 min)
import os, sys, glob, time, json, shutil, zipfile, subprocess, traceback
import numpy as np, cv2, torch
import matplotlib.pyplot as plt
from google.colab import drive, files

T_START = time.time()
drive.mount('/content/drive')
assert os.path.isdir('/content/drive/MyDrive'), 'Drive is not mounted'
assert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU'
BASE, OUT = '/content/drive/MyDrive/adp_project', '/content/bdd_subset'
if not os.path.isdir(f'{OUT}/labels'):
    zipfile.ZipFile(f'{BASE}/data/bdd100k_subset.zip').extractall(OUT)
W = f'{BASE}/runs/yolo11s_bdd/weights/best.pt'
assert os.path.exists(W), 'best.pt not found in Drive'
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'ultralytics', 'onnx', 'onnxslim'], check=True)
import ultralytics
from ultralytics import YOLO
os.makedirs('/content/exp', exist_ok=True)
PT = '/content/exp/best.pt'
shutil.copy(W, PT)
MODELS = f'{BASE}/models/yolo11s_bdd'; os.makedirs(MODELS, exist_ok=True)
errors, env = {}, dict(torch=torch.__version__, ultralytics=ultralytics.__version__, gpu=torch.cuda.get_device_name(0))

# ---------- exports ----------
onnx_path = engine_path = None
try:
    onnx_path = YOLO(PT).export(format='onnx', imgsz=640, batch=1, simplify=True, device=0)
    shutil.copy(onnx_path, f'{MODELS}/best.onnx')
    print('ONNX exported:', onnx_path, round(os.path.getsize(onnx_path) / 1e6, 1), 'MB')
except Exception as e:
    errors['onnx_export'] = f'{type(e).__name__}: {str(e)[:300]}'; print('ONNX export failed:', errors['onnx_export'])
try:
    engine_path = YOLO(PT).export(format='engine', half=True, imgsz=640, batch=1, device=0)
    shutil.copy(engine_path, f'{MODELS}/best_fp16_t4.engine')
    print('TensorRT engine built:', engine_path, round(os.path.getsize(engine_path) / 1e6, 1), 'MB')
except Exception as e:
    errors['tensorrt_export'] = f'{type(e).__name__}: {str(e)[:300]}'; print('TensorRT export failed:', errors['tensorrt_export'])
for pkg in ('tensorrt', 'onnxruntime'):
    try:
        env[pkg] = __import__(pkg).__version__
    except Exception:
        env[pkg] = 'not importable'
try:
    import onnxruntime as ort; env['onnxruntime_providers_available'] = ort.get_available_providers()
except Exception:
    pass
print('environment:', env)

In [ ]:
import os, sys, glob, time, json, shutil, zipfile, subprocess
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive, files

T_START = time.time()
drive.mount('/content/drive')
assert os.path.isdir('/content/drive/MyDrive'), 'Drive is not mounted'
BASE, OUT = '/content/drive/MyDrive/adp_project', '/content/bdd_subset'
MODELS = f'{BASE}/models/yolo11s_bdd'
ONNX, ENGINE = f'{MODELS}/best.onnx', f'{MODELS}/best_fp16_t4.engine'
for p in (ONNX, ENGINE, f'{BASE}/runs/yolo11s_bdd/weights/best.pt'):
    assert os.path.exists(p), f'missing in Drive: {p}'
os.makedirs('/content/exp', exist_ok=True)
PT = '/content/exp/best.pt'
if not os.path.exists(PT):
    shutil.copy(f'{BASE}/runs/yolo11s_bdd/weights/best.pt', PT)
if not os.path.isdir(f'{OUT}/labels'):
    zipfile.ZipFile(f'{BASE}/data/bdd100k_subset.zip').extractall(OUT)
try:
    import ultralytics, tensorrt, onnxruntime
except Exception as e:
    print('re-installing packages (runtime was recycled):', type(e).__name__)
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'ultralytics', 'onnx', 'onnxslim',
                    'onnxruntime-gpu', 'tensorrt-cu13==11.4.0.106'], check=True)
    import ultralytics, tensorrt, onnxruntime
import torch
assert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU'
env = dict(torch=torch.__version__, ultralytics=ultralytics.__version__, tensorrt=tensorrt.__version__,
           onnxruntime=onnxruntime.__version__, gpu=torch.cuda.get_device_name(0),
           ort_providers=onnxruntime.get_available_providers())
print('environment:', env)

WORKER = r'''
import sys, os, json, time, glob
import numpy as np, cv2, torch
from ultralytics import YOLO
label, weights, device, n, do_acc, out = sys.argv[1:7]
device = 0 if device == "0" else "cpu"
n = int(n); do_acc = do_acc == "1"
D = "/content/bdd_subset"
imgs = [cv2.imread(p) for p in sorted(glob.glob(D + "/images/test/*.jpg"))[: n + 15]]
m = YOLO(weights, task="detect")
inf, e2e = [], []
for k, im in enumerate(imgs):
    if device != "cpu": torch.cuda.synchronize()
    t0 = time.perf_counter()
    r = m.predict(im, imgsz=640, conf=0.25, device=device, verbose=False)[0]
    if device != "cpu": torch.cuda.synchronize()
    dt = (time.perf_counter() - t0) * 1000
    if k >= 15:
        inf.append(r.speed["inference"]); e2e.append(dt)
row = dict(label=label, images_timed=len(inf), model_ms_median=round(float(np.median(inf)), 2),
           model_ms_p95=round(float(np.percentile(inf, 95)), 2), total_ms_median=round(float(np.median(e2e)), 2),
           total_fps=round(1000 / float(np.median(e2e)), 1), size_mb=round(os.path.getsize(weights) / 1e6, 1))
try:
    pv = list(m.predictor.model.session.get_providers())
    row["onnx_providers_used"] = pv
    if device != "cpu" and "CUDAExecutionProvider" not in pv:
        row["label"] += "  [GPU requested but ran on CPU]"
except Exception:
    pass
if do_acc:
    v = YOLO(weights, task="detect").val(data=D + "/data.yaml", split="test", imgsz=640, batch=1, device=device,
                                         plots=False, verbose=False, project="/content/val_runs",
                                         name="w" + str(int(time.time())), exist_ok=True)
    row.update(test_mAP50=round(float(v.box.map50), 4), test_mAP50_95=round(float(v.box.map), 4),
               precision=round(float(v.box.mp), 4), recall=round(float(v.box.mr), 4))
json.dump(row, open(out, "w"))
'''
open('/content/bench_worker.py', 'w').write(WORKER)

rows = [
 dict(label='PyTorch FP32 (GPU)', images_timed=200, model_ms_median=9.42, model_ms_p95=16.33, total_ms_median=13.34,
      total_fps=74.9, size_mb=19.2, test_mAP50=0.4136, test_mAP50_95=0.2196, precision=0.7026, recall=0.3814, source='first run'),
 dict(label='PyTorch FP16 (GPU)', images_timed=200, model_ms_median=9.70, model_ms_p95=12.83, total_ms_median=13.49,
      total_fps=74.1, size_mb=19.2, test_mAP50=0.4135, test_mAP50_95=0.2194, precision=0.697, recall=0.3816, source='first run'),
]
errors = {}
PLAN = [('TensorRT FP16 (GPU)', ENGINE, '0', 200, 1),
        ('ONNX Runtime (CPU)', ONNX, 'cpu', 40, 1),
        ('PyTorch FP32 (CPU)', PT, 'cpu', 40, 0),
        ('ONNX Runtime (GPU)', ONNX, '0', 200, 0)]
for i, (label, wpath, dev, n, acc) in enumerate(PLAN):
    outj = f'/content/row_{i}.json'
    if os.path.exists(outj): os.remove(outj)
    t = time.time()
    try:
        r = subprocess.run([sys.executable, '/content/bench_worker.py', label, wpath, dev, str(n), str(acc), outj],
                           capture_output=True, text=True, timeout=1500)
        if r.returncode != 0 or not os.path.exists(outj):
            errors[label] = f'exit code {r.returncode} (negative = the process crashed). Last output: ' + (r.stderr or r.stdout)[-500:]
            print('FAILED:', label, '|', errors[label][:300])
        else:
            row = json.load(open(outj)); row['source'] = 'second run'; rows.append(row); print(f'{time.time()-t:.0f}s', row)
    except subprocess.TimeoutExpired:
        errors[label] = 'timed out after 25 minutes'; print('FAILED:', label, errors[label])

base_gpu, base_cpu = rows[0], next((r for r in rows if r['label'].startswith('PyTorch FP32 (CPU)')), None)
print('\n=== SUMMARY (T4 GPU / Colab CPU, batch 1, 640 px) ===')
for r in rows:
    ref = base_cpu if '(CPU)' in r['label'] else base_gpu
    sp = f"{ref['model_ms_median'] / r['model_ms_median']:.2f}x" if ref else 'n/a'
    print(f"{r['label']:<48} model {r['model_ms_median']:>7.2f} ms (p95 {r['model_ms_p95']:>7.2f}) | total {r['total_ms_median']:>7.2f} ms = {r['total_fps']:>6.1f} FPS | "
          f"mAP50 {r.get('test_mAP50', '-')} | mAP50-95 {r.get('test_mAP50_95', '-')} | model-time speedup {sp}")
if errors: print('\nERRORS:', json.dumps(errors, indent=1))

result = dict(environment=env, rows=rows, errors=errors,
              notes='first two rows come from the first run in the same VM; same 220 test images, batch 1, 640 px, warm-up 15 images; '
                    'CPU rows use 40 images on Colab CPU (not the laptop); engine built on and valid for this T4 only; '
                    'total time includes pre/post-processing; test-split mAP on a 800-image subset')
os.makedirs(f'{BASE}/logs', exist_ok=True)
json.dump(result, open(f'{BASE}/logs/benchmark_yolo.json', 'w'), indent=1)
fig, ax = plt.subplots(figsize=(9, 4.2))
lab = [r['label'].replace('  [GPU requested but ran on CPU]', '*') for r in rows]
ax.barh(lab[::-1], [r['model_ms_median'] for r in rows][::-1], color='#2b6cb0'); ax.set_xscale('log')
ax.set_xlabel('model time per image, median (ms, log scale)'); ax.set_title('YOLO11s latency by format (batch 1, 640 px)')
for i, r in enumerate(rows[::-1]):
    ax.text(r['model_ms_median'] * 1.05, i, f"{r['model_ms_median']:.1f} ms", va='center', fontsize=8)
plt.tight_layout(); plt.savefig('/content/benchmark_yolo.png', dpi=110); plt.show()
files.download('/content/benchmark_yolo.png'); files.download(f'{BASE}/logs/benchmark_yolo.json')
print(f'\nTotal time for this cell: {(time.time() - T_START) / 60:.1f} min')